# 🔴 Google Data Engineer – Page 4 (Q1112-Q1489)
| # | Title | Category | Difficulty |
|---|-------|----------|-----------|
| Q1119 | Speaker Totals by Language | SQL | EASY |
| Q1180 | User Activity Counts | SQL | MEDIUM |
| Q1267 | Latest vs Prior Ratings | SQL | HARD |
| Q1306 | Longest Increasing Subsequence | Python | MEDIUM |
| Q1342 | Count Multilingual Users | SQL | MEDIUM |
| Q1456 | Event Type Summary | SQL | MEDIUM |
| Q1460 | Merge Intervals | Python | MEDIUM |
| Q1463 | Avg Trip Time by Zip | SQL | HARD |


---
## Q1119 · Speaker Totals by Language (Database – EASY)

### Problem
Given `speakers` (user_id, language), count total speakers per language, sorted descending.

---
### 🧠 How to Remember
```
SELECT language, COUNT(*) → GROUP BY language → ORDER BY DESC
Simple! Just COUNT + GROUP BY
```


In [ ]:
import sqlite3, pandas as pd

conn4 = sqlite3.connect(":memory:")
conn4.execute("CREATE TABLE speakers (user_id INT, language TEXT)")
conn4.executemany("INSERT INTO speakers VALUES (?,?)", [
    (1,'English'),(2,'English'),(3,'Spanish'),(4,'Spanish'),(5,'Spanish'),
    (6,'French'),(7,'English'),(8,'Mandarin'),(9,'French'),(10,'Mandarin'),
    (11,'Mandarin'),(12,'Mandarin'),
])

# Solution 1: Simple GROUP BY
sql_spk1 = """
SELECT language, COUNT(*) AS speaker_count
FROM speakers
GROUP BY language
ORDER BY speaker_count DESC
"""
print("Solution 1 - Simple COUNT:")
print(pd.read_sql(sql_spk1, conn4))


In [ ]:
# Solution 2: With percentage
sql_spk2 = """
SELECT language, COUNT(*) AS speaker_count,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_share
FROM speakers
GROUP BY language
ORDER BY speaker_count DESC
"""
print("Solution 2 - With % share:")
print(pd.read_sql(sql_spk2, conn4))


In [ ]:
# Solution 3: Cumulative rank
sql_spk3 = """
WITH lang_counts AS (
    SELECT language, COUNT(*) AS cnt FROM speakers GROUP BY language
)
SELECT language, cnt AS speaker_count,
       RANK() OVER (ORDER BY cnt DESC) AS popularity_rank,
       SUM(cnt) OVER (ORDER BY cnt DESC) AS cumulative_speakers
FROM lang_counts
"""
print("Solution 3 - With cumulative count:")
print(pd.read_sql(sql_spk3, conn4))


---
## Q1267 · Latest vs Prior Ratings (Database – HARD)

### Problem
Given `ratings` (user_id, product_id, stars, submitted_at), for each user+product,
compare the LATEST rating to the PRIOR (second latest) rating.
Return rows where latest > prior.

---
### 🧠 How to Remember
```
SELF-JOIN or LAG() window function:
1. LAG(stars) OVER (PARTITION BY user_id, product_id ORDER BY submitted_at)
2. Filter WHERE current > lag_value
3. Or: ROW_NUMBER = 1 (latest) and ROW_NUMBER = 2 (prior) → JOIN on same user+product
```


In [ ]:
conn4.execute("""
CREATE TABLE ratings (user_id INT, product_id INT, stars INT, submitted_at TEXT)
""")
conn4.executemany("INSERT INTO ratings VALUES (?,?,?,?)", [
    (1, 10, 3, '2023-01-01'),
    (1, 10, 4, '2023-02-01'),
    (1, 10, 5, '2023-03-01'),   # latest for user1+prod10
    (2, 10, 5, '2023-01-01'),
    (2, 10, 3, '2023-02-01'),   # latest for user2+prod10 (decreased)
    (3, 20, 4, '2023-01-01'),
    (3, 20, 5, '2023-02-01'),   # latest for user3+prod20
])

# Solution 1: LAG() window function
sql_rating1 = """
WITH lagged AS (
    SELECT user_id, product_id, stars,
           LAG(stars) OVER (PARTITION BY user_id, product_id ORDER BY submitted_at) AS prior_stars,
           submitted_at,
           ROW_NUMBER() OVER (PARTITION BY user_id, product_id ORDER BY submitted_at DESC) AS rn
    FROM ratings
)
SELECT user_id, product_id, stars AS latest_rating, prior_stars AS prior_rating
FROM lagged
WHERE rn = 1 AND prior_stars IS NOT NULL AND stars > prior_stars
"""
print("Solution 1 - LAG() function:")
print(pd.read_sql(sql_rating1, conn4))


In [ ]:
# Solution 2: Self-JOIN with ROW_NUMBER
sql_rating2 = """
WITH ranked AS (
    SELECT user_id, product_id, stars, submitted_at,
           ROW_NUMBER() OVER (PARTITION BY user_id, product_id ORDER BY submitted_at DESC) AS rn
    FROM ratings
)
SELECT a.user_id, a.product_id,
       a.stars AS latest_rating, b.stars AS prior_rating,
       a.stars - b.stars AS improvement
FROM ranked a
JOIN ranked b ON a.user_id = b.user_id
              AND a.product_id = b.product_id
              AND b.rn = 2
WHERE a.rn = 1 AND a.stars > b.stars
ORDER BY a.user_id, a.product_id
"""
print("Solution 2 - Self-JOIN approach:")
print(pd.read_sql(sql_rating2, conn4))


In [ ]:
# Solution 3: Full comparison report (all users)
sql_rating3 = """
WITH ranked AS (
    SELECT user_id, product_id, stars,
           ROW_NUMBER() OVER (PARTITION BY user_id, product_id ORDER BY submitted_at DESC) AS rn
    FROM ratings
),
latest AS (SELECT user_id, product_id, stars FROM ranked WHERE rn=1),
prior  AS (SELECT user_id, product_id, stars FROM ranked WHERE rn=2)
SELECT l.user_id, l.product_id,
       l.stars AS latest, p.stars AS prior,
       CASE WHEN l.stars > p.stars THEN 'Improved'
            WHEN l.stars < p.stars THEN 'Declined'
            ELSE 'Same' END AS trend
FROM latest l
JOIN prior p ON l.user_id=p.user_id AND l.product_id=p.product_id
ORDER BY l.user_id
"""
print("Solution 3 - Full trend report:")
print(pd.read_sql(sql_rating3, conn4))


---
## Q1306 · Longest Increasing Subsequence (Algorithms – MEDIUM)

### Problem
Given an integer array, find the length of the longest strictly increasing subsequence.

**Example:** [10,9,2,5,3,7,101,18] → 4  (2,3,7,101 or 2,5,7,101)

---
### 🧠 How to Remember (Step by Step)
```
THREE approaches - know all three!
1. DP O(n²): dp[i] = max(dp[j]+1) for all j<i where nums[j]<nums[i]
2. Binary Search O(n log n): maintain 'tails' array
   - tails[i] = smallest tail of all IS of length i+1
   - Binary search to find where new element fits
3. Patience Sorting insight: same as #2, intuition from card game
```


In [ ]:
import bisect

# Solution 1: DP O(n²)
# Time: O(n²)  Space: O(n)
def lis_dp(nums: List[int]) -> int:
    """dp[i] = LIS ending at index i."""
    if not nums: return 0
    n = len(nums)
    dp = [1] * n

    for i in range(1, n):
        for j in range(i):
            if nums[j] < nums[i]:
                dp[i] = max(dp[i], dp[j] + 1)

    return max(dp)

nums = [10,9,2,5,3,7,101,18]
print("Solution 1 (DP O(n²)):", lis_dp(nums))  # 4


In [ ]:
# Solution 2: Binary Search (Patience Sorting)
# Time: O(n log n)  Space: O(n)
def lis_binary_search(nums: List[int]) -> int:
    """
    tails[i] = smallest ending element of all IS of length i+1.
    Binary search to find position for each new element.
    """
    tails = []

    for num in nums:
        pos = bisect.bisect_left(tails, num)  # where does num fit?
        if pos == len(tails):
            tails.append(num)      # extend LIS
        else:
            tails[pos] = num       # replace to keep tails optimal

    return len(tails)

print("Solution 2 (Binary Search):", lis_binary_search(nums))  # 4
print("  tails trace:", end=" ")
tails = []
for n2 in nums:
    pos = bisect.bisect_left(tails, n2)
    if pos == len(tails): tails.append(n2)
    else: tails[pos] = n2
    print(tails, end=" → ")
print()


In [ ]:
# Solution 3: Reconstruct actual LIS (not just length)
# Time: O(n log n)  Space: O(n)
def lis_reconstruct(nums: List[int]):
    """Return the actual subsequence, not just length."""
    tails = []
    parent = [-1] * len(nums)
    index_map = []  # maps tail position to nums index

    for i, num in enumerate(nums):
        pos = bisect.bisect_left(tails, num)
        if pos == len(tails):
            tails.append(num)
            index_map.append(i)
        else:
            tails[pos] = num
            index_map[pos] = i
        if pos > 0:
            parent[i] = index_map[pos-1]

    # Reconstruct path
    seq = []
    idx = index_map[-1]
    while idx != -1:
        seq.append(nums[idx])
        idx = parent[idx]
    return list(reversed(seq))

print("Solution 3 (Reconstruct LIS):", lis_reconstruct(nums))
print("\n⏱ Complexity Summary:")
print("| Solution         | Time      | Space |")
print("|------------------|-----------|-------|")
print("| DP               | O(n²)     | O(n)  |")
print("| Binary Search ✅ | O(n log n)| O(n)  |")
print("| Reconstruct      | O(n log n)| O(n)  |")


---
## Q1460 · Merge Intervals (Data Structures – MEDIUM)

### Problem
Given array of [start,end] intervals, merge all overlapping intervals.

**Example:** [[1,3],[2,6],[8,10],[15,18]] → [[1,6],[8,10],[15,18]]

---
### 🧠 How to Remember
```
1. SORT by start time
2. Init result with first interval
3. For each interval: if start <= result[-1][1] → EXTEND (merge)
                       else → APPEND new interval
Key: "If new start ≤ last end → overlap → extend the end"
```


In [ ]:
# Solution 1: Sort + Linear Merge (Standard)
# Time: O(n log n)  Space: O(n)
def merge_intervals(intervals: List[List[int]]) -> List[List[int]]:
    intervals.sort(key=lambda x: x[0])
    merged = [intervals[0]]

    for start, end in intervals[1:]:
        if start <= merged[-1][1]:           # overlap
            merged[-1][1] = max(merged[-1][1], end)
        else:
            merged.append([start, end])
    return merged

print("Solution 1:", merge_intervals([[1,3],[2,6],[8,10],[15,18]]))
print("Solution 1:", merge_intervals([[1,4],[4,5]]))  # [[1,5]]


In [ ]:
# Solution 2: Using stack
# Time: O(n log n)  Space: O(n)
def merge_intervals_stack(intervals: List[List[int]]) -> List[List[int]]:
    intervals.sort()
    stack = []
    for start, end in intervals:
        if stack and start <= stack[-1][1]:
            stack[-1][1] = max(stack[-1][1], end)
        else:
            stack.append([start, end])
    return stack

print("Solution 2:", merge_intervals_stack([[1,3],[2,6],[8,10],[15,18]]))


In [ ]:
# Solution 3: Functional / one-liner style
# Time: O(n log n)  Space: O(n)
def merge_intervals_func(intervals: List[List[int]]) -> List[List[int]]:
    """Reduce-based merge."""
    from functools import reduce

    def merge_two(acc, curr):
        if acc and acc[-1][1] >= curr[0]:
            acc[-1] = [acc[-1][0], max(acc[-1][1], curr[1])]
        else:
            acc.append(list(curr))
        return acc

    return reduce(merge_two, sorted(intervals), [])

print("Solution 3:", merge_intervals_func([[1,3],[2,6],[8,10],[15,18]]))
print("\n⏱ Complexity Summary:")
print("All solutions: Time O(n log n), Space O(n)")


---
## Q1463 · Avg Trip Time by Zip (Database – HARD)

### Problem
Given `trips` (trip_id, pickup_zip, dropoff_zip, pickup_ts, dropoff_ts),
compute average trip duration in minutes for each zip code pair.
Only include pairs with at least 3 trips.

---
### 🧠 How to Remember
```
1. Calculate duration per trip: (dropoff - pickup) in minutes
2. GROUP BY pickup_zip, dropoff_zip
3. AVG(duration)
4. HAVING COUNT(*) >= 3
```


In [ ]:
conn4.execute("""
CREATE TABLE trips (
    trip_id INT, pickup_zip TEXT, dropoff_zip TEXT,
    pickup_ts TEXT, dropoff_ts TEXT
)
""")
conn4.executemany("INSERT INTO trips VALUES (?,?,?,?,?)", [
    (1,'94105','94102','2023-01-01 10:00','2023-01-01 10:15'),
    (2,'94105','94102','2023-01-01 11:00','2023-01-01 11:20'),
    (3,'94105','94102','2023-01-01 12:00','2023-01-01 12:18'),
    (4,'94102','94105','2023-01-01 13:00','2023-01-01 13:25'),
    (5,'94102','94105','2023-01-01 14:00','2023-01-01 14:22'),
    (6,'94102','94105','2023-01-01 15:00','2023-01-01 15:28'),
    (7,'94105','94107','2023-01-01 16:00','2023-01-01 16:30'),
    (8,'94105','94107','2023-01-01 17:00','2023-01-01 17:35'),
])

# Solution 1: strftime duration + GROUP BY HAVING
sql_trip1 = """
SELECT pickup_zip, dropoff_zip,
       COUNT(*) AS trip_count,
       ROUND(AVG(
           (strftime('%s', dropoff_ts) - strftime('%s', pickup_ts)) / 60.0
       ), 1) AS avg_duration_min
FROM trips
GROUP BY pickup_zip, dropoff_zip
HAVING COUNT(*) >= 3
ORDER BY avg_duration_min DESC
"""
print("Solution 1 - AVG duration (min 3 trips):")
print(pd.read_sql(sql_trip1, conn4))


In [ ]:
# Solution 2: CTE with explicit duration calc + percentiles
sql_trip2 = """
WITH durations AS (
    SELECT pickup_zip, dropoff_zip,
           (strftime('%s', dropoff_ts) - strftime('%s', pickup_ts)) / 60.0 AS duration_min
    FROM trips
)
SELECT pickup_zip, dropoff_zip,
       COUNT(*) AS trip_count,
       ROUND(AVG(duration_min), 1) AS avg_min,
       ROUND(MIN(duration_min), 1) AS min_min,
       ROUND(MAX(duration_min), 1) AS max_min
FROM durations
GROUP BY pickup_zip, dropoff_zip
HAVING COUNT(*) >= 2   -- relaxed for demo
ORDER BY avg_min
"""
print("Solution 2 - With min/max duration:")
print(pd.read_sql(sql_trip2, conn4))


In [ ]:
# Solution 3: Pandas equivalent
df_trips = pd.read_sql("SELECT * FROM trips", conn4)
df_trips['pickup_ts'] = pd.to_datetime(df_trips['pickup_ts'])
df_trips['dropoff_ts'] = pd.to_datetime(df_trips['dropoff_ts'])
df_trips['duration_min'] = (df_trips['dropoff_ts'] - df_trips['pickup_ts']).dt.total_seconds() / 60

grouped = df_trips.groupby(['pickup_zip','dropoff_zip']).agg(
    trip_count=('trip_id','count'),
    avg_duration=('duration_min','mean')
).reset_index()
print("Solution 3 - Pandas:")
print(grouped[grouped['trip_count'] >= 2].round(1))
